<a href="https://colab.research.google.com/github/nirjanashrestha4/Mr604Nirjana/blob/main/mr604Nirjana.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Check what GPU we have.
This tells us what computer power we are using for training

In [ ]:
!nvidia-smi

# Connect Google Drive to this notebook.
This lets us save our trained model and results permanently.
Without this everything is deleted when the session ends.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

#Check all dataset folders are in Google Drive

In [ ]:
import os

paths = [
    '/content/drive/MyDrive/cvaf_swinb/cbis/csv',
    '/content/drive/MyDrive/cvaf_swinb/cbis/jpeg',
    '/content/drive/MyDrive/cvaf_swinb/vindr/images',
    '/content/drive/MyDrive/cvaf_swinb/vindr/annotations',
]

for path in paths:
    if os.path.exists(path):
        count = len(os.listdir(path))
        print(f"OK      {path}  ({count} items)")
    else:
        print(f"MISSING {path}")

# Check CSV files for CBIS-DDSM

In [ ]:
base = '/content/drive/MyDrive/cvaf_swinb'

print("=== CBIS CSV files ===")
for f in sorted(os.listdir(f'{base}/cbis/csv')):
    size = os.path.getsize(f'{base}/cbis/csv/{f}') / 1024
    print(f"  {f}  ({size:.0f} KB)")

#Check JPEG structure for CBIS-DDSM

In [ ]:
print("\n=== CBIS JPEG (first 5 folders) ===")
jpeg_items = sorted(os.listdir(f'{base}/cbis/jpeg'))[:5]
for f in jpeg_items:
    full = f'{base}/cbis/jpeg/{f}'
    if os.path.isdir(full):
        count = len(os.listdir(full))
        print(f"  {f}/  ({count} files)")
    else:
        print(f"  {f}")

# Load the mass train file.

This file contains information about mass cases in CBIS-DDSM.

Mass = a lump found in the breast

In [ ]:
import pandas as pd

mass_train = pd.read_csv(f'{base}/cbis/csv/mass_case_description_train_set.csv')

#Show basic info


In [ ]:
print("Number of rows    :", len(mass_train))
print("Number of columns :", len(mass_train.columns))

#Show all column names.
This tells us what information is stored in this file

In [ ]:
print("Column names:")
for i, col in enumerate(mass_train.columns):
    print(f"  {i+1}. {col}")

#Show first 5 rows.
This lets us see actual data values in each column

In [ ]:
print(mass_train.head())

#Keep only the columns we need.
 Drop everything else

In [ ]:
columns_to_keep = [
    'patient_id',
    'left or right breast',
    'image view',
    'pathology',
    'image file path'
]

mass_train_clean = mass_train[columns_to_keep].copy()

print("Before:", mass_train.shape)
print("After :", mass_train_clean.shape)
print()
print("First 5 rows:")
print(mass_train_clean.head())

#Check for missing values.
Missing values = empty cells in our data. We need to handle these before training

In [ ]:
print("Missing values in each column:")
print(mass_train_clean.isnull().sum())
print()
print("Total missing values:", mass_train_clean.isnull().sum().sum())

# Load mass_test file

In [ ]:
mass_test = pd.read_csv(f'{base}/cbis/csv/mass_case_description_test_set.csv')


#Show basic info


In [ ]:
print("Number of rows    :", len(mass_test))
print("Number of columns :", len(mass_test.columns))

# Show all column names


In [ ]:
print("Column names:")
for i, col in enumerate(mass_test.columns):
    print(f"  {i+1}. {col}")

# Show first 5 rows

In [ ]:
print(mass_test.head())

#Keep only the columns we need

In [ ]:
mass_test_clean = mass_test[columns_to_keep].copy()

print("Before:", mass_test.shape)
print("After :", mass_test_clean.shape)
print()
print("First 5 rows:")
print(mass_test_clean.head())

#Check for missing values in mass_test

In [ ]:
print("Missing values in each column:")
print(mass_test_clean.isnull().sum())
print()
print("Total missing values:", mass_test_clean.isnull().sum().sum())

# Load calc_train and calc_test

In [ ]:
calc_train = pd.read_csv(f'{base}/cbis/csv/calc_case_description_train_set.csv')
calc_test  = pd.read_csv(f'{base}/cbis/csv/calc_case_description_test_set.csv')

#Keep only columns we need

In [ ]:
calc_train_clean = calc_train[columns_to_keep].copy()
calc_test_clean  = calc_test[columns_to_keep].copy()

#Check both files

In [ ]:
for name, df in [
    ('calc_train', calc_train_clean),
    ('calc_test',  calc_test_clean),
]:
    print(f"--- {name} ---")
    print(f"  Rows           : {len(df)}")
    print(f"  Columns        : {len(df.columns)}")
    print(f"  Missing values : {df.isnull().sum().sum()}")
    print()

#Check class distribution for all 4 files


In [ ]:
for name, df in [
    ('mass_train', mass_train_clean),
    ('mass_test',  mass_test_clean),
    ('calc_train', calc_train_clean),
    ('calc_test',  calc_test_clean),
]:
    print(f"--- {name} ---")
    for label, count in df['pathology'].value_counts().items():
        pct = count / len(df) * 100
        print(f"  {label:35s} : {count:4d}  ({pct:.1f}%)")
    print()

# Convert pathology to binary label
MALIGNANT              → 1 (cancer)

BENIGN                 → 0 (not cancer)

BENIGN_WITHOUT_CALLBACK → 0 (not cancer)

In [ ]:
for name, df in [
    ('mass_train', mass_train_clean),
    ('mass_test',  mass_test_clean),
    ('calc_train', calc_train_clean),
    ('calc_test',  calc_test_clean),
]:
    df['label'] = df['pathology'].map({
        'MALIGNANT':               1,
        'BENIGN':                  0,
        'BENIGN_WITHOUT_CALLBACK': 0,
    })

    print(f"=== {name} ===")
    print(f"  Benign (0)    : {(df['label']==0).sum()}")
    print(f"  Malignant (1) : {(df['label']==1).sum()}")
    print()

# Combine all 4 files into one dataframe for CBIS-DDSM


In [ ]:
cbis_df = pd.concat([
    mass_train_clean,
    mass_test_clean,
    calc_train_clean,
    calc_test_clean
], ignore_index=True)

print("Combined CBIS-DDSM dataset:")
print(f"  Total rows    : {len(cbis_df)}")
print(f"  Benign (0)    : {(cbis_df['label']==0).sum()}")
print(f"  Malignant (1) : {(cbis_df['label']==1).sum()}")
print(f"  Ratio         : {(cbis_df['label']==0).sum() / (cbis_df['label']==1).sum():.1f}:1")

#Check VinDr annotations

In [ ]:
print("\n=== VinDr annotation files ===")
for f in sorted(os.listdir(f'{base}/vindr/annotations')):
    size = os.path.getsize(f'{base}/vindr/annotations/{f}') / 1024
    print(f"  {f}  ({size:.0f} KB)")

#Check VinDr images

In [ ]:
print("\n=== VinDr images (first 5) ===")
vindr_items = sorted(os.listdir(f'{base}/vindr/images'))[:5]
for f in vindr_items:
    print(f"  {f}")

# Load VinDr breast level annotations

In [ ]:
vindr_df = pd.read_csv(f'{base}/vindr/annotations/breast-level_annotations.csv')

print("Number of rows    :", len(vindr_df))
print("Number of columns :", len(vindr_df.columns))

# Show all column names


In [ ]:
print("Column names:")
for i, col in enumerate(vindr_df.columns):
    print(f"  {i+1}. {col}")

# Show first 5 rows


In [ ]:
print(vindr_df.head())

# Columns we need from VinDr
study_id      = patient identifier

 image_id      = to find the image file

 laterality    = LEFT or RIGHT breast

view_position = CC or MLO

breast_birads = our label (BI-RADS score)



In [ ]:
vindr_columns_to_keep = [
    'study_id',
    'image_id',
    'laterality',
    'view_position',
    'breast_birads'
]

vindr_clean = vindr_df[vindr_columns_to_keep].copy()

print("Before:", vindr_df.shape)
print("After :", vindr_clean.shape)
print()
print("First 5 rows:")
print(vindr_clean.head())

# Check for missing values in VinDr


In [ ]:
print("Missing values in each column:")
print(vindr_clean.isnull().sum())
print()
print("Total missing values:", vindr_clean.isnull().sum().sum())

# Check BI-RADS distribution
BI-RADS is the scoring system radiologists use

BI-RADS 1, 2, 3 = Benign (not cancer)

BI-RADS 4, 5    = Malignant (suspicious for cancer)

 BI-RADS 0       = Incomplete (needs more images)


In [ ]:
print("BI-RADS distribution:")
print(vindr_clean['breast_birads'].value_counts().sort_index())

# Convert BI-RADS to binary label
BI-RADS 1, 2, 3 → Benign (0)

BI-RADS 4, 5    → Malignant (1)

BI-RADS 0       → removed (incomplete)



In [ ]:
vindr_clean['label'] = vindr_clean['breast_birads'].map({
    'BI-RADS 1': 0,
    'BI-RADS 2': 0,
    'BI-RADS 3': 0,
    'BI-RADS 4': 1,
    'BI-RADS 5': 1,
})

# Remove rows where label is NaN (BI-RADS 0)


In [ ]:
vindr_clean = vindr_clean.dropna(subset=['label'])
vindr_clean['label'] = vindr_clean['label'].astype(int)

print("VinDr after label conversion:")
print(f"  Total rows    : {len(vindr_clean)}")
print(f"  Benign (0)    : {(vindr_clean['label']==0).sum()}")
print(f"  Malignant (1) : {(vindr_clean['label']==1).sum()}")
print(f"  Ratio         : {(vindr_clean['label']==0).sum() / (vindr_clean['label']==1).sum():.1f}:1")

# Show class imbalance summary for both datasets


In [ ]:
print("="*50)
print("CLASS IMBALANCE SUMMARY")
print("="*50)
print()
print("CBIS-DDSM:")
print(f"  Benign (0)    : {(cbis_df['label']==0).sum()}")
print(f"  Malignant (1) : {(cbis_df['label']==1).sum()}")
print(f"  Ratio         : {(cbis_df['label']==0).sum() / (cbis_df['label']==1).sum():.1f}:1")
print()
print("VinDr-Mammo:")
print(f"  Benign (0)    : {(vindr_clean['label']==0).sum()}")
print(f"  Malignant (1) : {(vindr_clean['label']==1).sum()}")
print(f"  Ratio         : {(vindr_clean['label']==0).sum() / (vindr_clean['label']==1).sum():.1f}:1")
print()
print("="*50)
print("Problem:")
print("  CBIS-DDSM  : 1.4:1  - fairly balanced")
print("  VinDr      : 19.2:1 - very imbalanced")
print("  This means the model will be biased")
print("  toward predicting Benign")
print("="*50)

# Plot class distribution for both datasets

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Class Distribution', fontsize=14, fontweight='bold')

# CBIS-DDSM plot
ax1 = axes[0]
cbis_counts = [
    (cbis_df['label']==0).sum(),
    (cbis_df['label']==1).sum()
]
bars1 = ax1.bar(['Benign', 'Malignant'], cbis_counts,
                color=['royalblue', 'tomato'], alpha=0.85)
ax1.set_title('CBIS-DDSM', fontsize=12, fontweight='bold')
ax1.set_ylabel('Number of cases')
for bar, val in zip(bars1, cbis_counts):
    ax1.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 10,
             f'{val}\n({val/sum(cbis_counts)*100:.1f}%)',
             ha='center', fontsize=11, fontweight='bold')
ax1.set_ylim([0, max(cbis_counts) * 1.2])
ax1.grid(alpha=0.3, axis='y')

# VinDr plot
ax2 = axes[1]
vindr_counts = [
    (vindr_clean['label']==0).sum(),
    (vindr_clean['label']==1).sum()
]
bars2 = ax2.bar(['Benign', 'Malignant'], vindr_counts,
                color=['royalblue', 'tomato'], alpha=0.85)
ax2.set_title('VinDr-Mammo', fontsize=12, fontweight='bold')
ax2.set_ylabel('Number of cases')
for bar, val in zip(bars2, vindr_counts):
    ax2.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 10,
             f'{val}\n({val/sum(vindr_counts)*100:.1f}%)',
             ha='center', fontsize=11, fontweight='bold')
ax2.set_ylim([0, max(vindr_counts) * 1.2])
ax2.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/cvaf_swinb/class_distribution.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved to Google Drive")

# Check how many unique views exist in VinDr

In [ ]:
print("Unique laterality values:")
print(vindr_clean['laterality'].unique())
print()
print("Unique view positions:")
print(vindr_clean['view_position'].unique())
print()

# Check exact view values in VinDr


In [ ]:
print("Laterality values:")
print(vindr_clean['laterality'].value_counts())
print()
print("View position values:")
print(vindr_clean['view_position'].value_counts())
print()

# Check one patient to see their views


In [ ]:
sample_id = vindr_clean['study_id'].iloc[0]
sample = vindr_clean[vindr_clean['study_id'] == sample_id]
print(f"Sample patient: {sample_id}")
print(sample[['laterality', 'view_position', 'label']])

# Fix laterality — convert L/R to LEFT/RIGHT


In [ ]:
vindr_clean['laterality'] = vindr_clean['laterality'].replace({
    'L': 'LEFT',
    'R': 'RIGHT'
})

print("Laterality after fix:")
print(vindr_clean['laterality'].value_counts())
print()

# Now check 4 views again


In [ ]:
required_views = {('LEFT','CC'), ('LEFT','MLO'), ('RIGHT','CC'), ('RIGHT','MLO')}

complete   = 0
incomplete = 0

for study_id, group in vindr_clean.groupby('study_id'):
    views = set(zip(group['laterality'], group['view_position']))
    if required_views.issubset(views):
        complete += 1
    else:
        incomplete += 1

print(f"Patients with all 4 views   : {complete}")
print(f"Patients with missing views : {incomplete}")

# Remove the 1 patient who does not have all 4 views
Find patients with all 4 views

In [ ]:
complete_ids = []
for study_id, group in vindr_clean.groupby('study_id'):
    views = set(zip(group['laterality'], group['view_position']))
    if required_views.issubset(views):
        complete_ids.append(study_id)



# Keep only complete patients

In [ ]:
vindr_clean = vindr_clean[vindr_clean['study_id'].isin(complete_ids)]

print(f"Patients after removing incomplete : {vindr_clean['study_id'].nunique()}")
print(f"Total rows                         : {len(vindr_clean)}")
print(f"Benign (0)                         : {(vindr_clean['label']==0).sum()}")
print(f"Malignant (1)                      : {(vindr_clean['label']==1).sum()}")

# Check 4 views in CBIS-DDSM
 First normalise the column names to match VinDr

In [ ]:

cbis_df['left or right breast'] = cbis_df['left or right breast'].str.upper().str.strip()
cbis_df['image view'] = cbis_df['image view'].str.upper().str.strip()

print("Unique sides in CBIS:")
print(cbis_df['left or right breast'].value_counts())
print()
print("Unique views in CBIS:")
print(cbis_df['image view'].value_counts())
print()

# Check how many patients have all 4 views

In [ ]:

required_views = {('LEFT','CC'), ('LEFT','MLO'), ('RIGHT','CC'), ('RIGHT','MLO')}

complete   = 0
incomplete = 0

for patient_id, group in cbis_df.groupby('patient_id'):
    views = set(zip(group['left or right breast'], group['image view']))
    if required_views.issubset(views):
        complete += 1
    else:
        incomplete += 1

print(f"Patients with all 4 views   : {complete}")
print(f"Patients with missing views : {incomplete}")

# Keep only patients with all 4 views


In [ ]:
required_views = {('LEFT','CC'), ('LEFT','MLO'), ('RIGHT','CC'), ('RIGHT','MLO')}

complete_ids = []
for patient_id, group in cbis_df.groupby('patient_id'):
    views = set(zip(group['left or right breast'], group['image view']))
    if required_views.issubset(views):
        complete_ids.append(patient_id)



# Keep only complete patients


In [ ]:
cbis_clean = cbis_df[cbis_df['patient_id'].isin(complete_ids)]

print(f"Patients with all 4 views : {cbis_clean['patient_id'].nunique()}")
print(f"Total rows                : {len(cbis_clean)}")
print(f"Benign (0)                : {(cbis_clean['label']==0).sum()}")
print(f"Malignant (1)             : {(cbis_clean['label']==1).sum()}")

# Combined summary of both datasets


In [ ]:
print("="*50)
print("DATASET SUMMARY")
print("="*50)
print()
print("CBIS-DDSM:")
print(f"  Patients  : {cbis_clean['patient_id'].nunique()}")
print(f"  Benign    : {(cbis_clean['label']==0).sum()}")
print(f"  Malignant : {(cbis_clean['label']==1).sum()}")
print()
print("VinDr-Mammo:")
print(f"  Patients  : {vindr_clean['study_id'].nunique()}")
print(f"  Benign    : {(vindr_clean['label']==0).sum()}")
print(f"  Malignant : {(vindr_clean['label']==1).sum()}")
print()
print("="*50)
print("COMBINED:")
total_b = (cbis_clean['label']==0).sum() + (vindr_clean['label']==0).sum()
total_m = (cbis_clean['label']==1).sum() + (vindr_clean['label']==1).sum()
total   = total_b + total_m
print(f"  Total patients : {cbis_clean['patient_id'].nunique() + vindr_clean['study_id'].nunique()}")
print(f"  Benign         : {total_b}")
print(f"  Malignant      : {total_m}")
print(f"  Ratio          : {total_b/total_m:.1f}:1")
print("="*50)

#Now let's visualise this:

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Dataset Summary', fontsize=14, fontweight='bold')

# CBIS-DDSM
ax1 = axes[0]
cbis_counts = [622, 83]
bars1 = ax1.bar(['Benign', 'Malignant'], cbis_counts,
                color=['royalblue', 'tomato'], alpha=0.85)
ax1.set_title('CBIS-DDSM\n(105 patients)', fontsize=11, fontweight='bold')
ax1.set_ylabel('Number of patients')
for bar, val in zip(bars1, cbis_counts):
    ax1.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 5,
             f'{val}\n({val/sum(cbis_counts)*100:.1f}%)',
             ha='center', fontsize=10, fontweight='bold')
ax1.set_ylim([0, max(cbis_counts) * 1.3])
ax1.grid(alpha=0.3, axis='y')

# VinDr
ax2 = axes[1]
vindr_counts = [19008, 988]
bars2 = ax2.bar(['Benign', 'Malignant'], vindr_counts,
                color=['royalblue', 'tomato'], alpha=0.85)
ax2.set_title('VinDr-Mammo\n(4999 patients)', fontsize=11, fontweight='bold')
ax2.set_ylabel('Number of patients')
for bar, val in zip(bars2, vindr_counts):
    ax2.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 50,
             f'{val}\n({val/sum(vindr_counts)*100:.1f}%)',
             ha='center', fontsize=10, fontweight='bold')
ax2.set_ylim([0, max(vindr_counts) * 1.3])
ax2.grid(alpha=0.3, axis='y')

# Combined
ax3 = axes[2]
combined_counts = [19630, 1071]
bars3 = ax3.bar(['Benign', 'Malignant'], combined_counts,
                color=['royalblue', 'tomato'], alpha=0.85)
ax3.set_title('Combined\n(5104 patients)', fontsize=11, fontweight='bold')
ax3.set_ylabel('Number of patients')
for bar, val in zip(bars3, combined_counts):
    ax3.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 50,
             f'{val}\n({val/sum(combined_counts)*100:.1f}%)',
             ha='center', fontsize=10, fontweight='bold')
ax3.set_ylim([0, max(combined_counts) * 1.3])
ax3.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/cvaf_swinb/dataset_summary.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved to Google Drive")

# Outlier detection
We scan each patient's L-CC image and check pixel statistics. A good mammogram image should have:
- Mean pixel intensity not too dark or too bright
- Standard deviation not too low (some contrast must exist)



In [ ]:
import os
import numpy as np
from PIL import Image

base = '/content/drive/MyDrive/cvaf_swinb'

# First index all CBIS jpeg images


In [50]:
print("Indexing CBIS images ...")
cbis_img_index = {}
jpeg_dir = f'{base}/cbis/jpeg'

for folder in os.listdir(jpeg_dir):
    folder_path = f'{jpeg_dir}/{folder}'
    if os.path.isdir(folder_path):
        for img_file in os.listdir(folder_path):
            if img_file.endswith('.jpg'):
                cbis_img_index[folder] = f'{folder_path}/{img_file}'

print(f"CBIS images indexed: {len(cbis_img_index)}")

Indexing CBIS images ...
CBIS images indexed: 6774


# Check what is inside VinDr images folder


In [51]:
vindr_img_dir = f'{base}/vindr/images'

items = os.listdir(vindr_img_dir)
print(f"Total items: {len(items)}")
print()
print("First 5 items:")
for item in items[:5]:
    full_path = f'{vindr_img_dir}/{item}'
    if os.path.isdir(full_path):
        count = len(os.listdir(full_path))
        print(f"  DIR  {item}/  ({count} files)")
    else:
        print(f"  FILE {item}")

Total items: 5001

First 5 items:
  DIR  d7169c8f349e5c37fc0cf2ce78cdd4cb/  (4 files)
  DIR  a32528da615a00d8a2a07deb4cc4cbc7/  (4 files)
  DIR  0ba0f88bfd64d856fb97f6efce9f27a4/  (4 files)
  DIR  8f70a359d0130a439c60ebf1f8dfdb1a/  (4 files)
  DIR  8f4cbcda4233497b56034740b8991c72/  (4 files)


# Check what is inside one VinDr image folder


In [52]:
sample_folder = f'{vindr_img_dir}/{items[0]}'
print(f"Folder: {items[0]}")
print()
print("Files inside:")
for f in os.listdir(sample_folder):
    print(f"  {f}")

Folder: d7169c8f349e5c37fc0cf2ce78cdd4cb

Files inside:
  57b34fd0da5ef7b8e9afc6b269ccf717.png
  91c76c5acb6cdf9e18869ef78c7d043a.png
  69b872a225cb0e218a26a8a313ee3ef4.png
  be320285b835f2429b96322830363c9f.png


# Index VinDr images correctly


Structure is: images/study_id/image_id.png

In [ ]:
print("Indexing VinDr images ...")
vindr_img_index = {}

for folder in os.listdir(vindr_img_dir):
    folder_path = f'{vindr_img_dir}/{folder}'
    if os.path.isdir(folder_path):
        for img_file in os.listdir(folder_path):
            if img_file.endswith('.png'):
                image_id = img_file.replace('.png', '')
                vindr_img_index[image_id] = f'{folder_path}/{img_file}'

print(f"VinDr images indexed: {len(vindr_img_index)}")
print()
print("Sample entry:")
sample_key = list(vindr_img_index.keys())[0]
print(f"  image_id : {sample_key}")
print(f"  path     : {vindr_img_index[sample_key]}")

Indexing VinDr images ...


# Match each row in VinDr CSV to its actual image file

In [ ]:
vindr_clean['img_path'] = vindr_clean['image_id'].map(vindr_img_index)

# Check how many rows got a path

In [ ]:
matched   = vindr_clean['img_path'].notna().sum()
unmatched = vindr_clean['img_path'].isna().sum()

print(f"Matched images   : {matched}")
print(f"Unmatched images : {unmatched}")
print()
print("Sample rows with image paths:")
print(vindr_clean[['study_id', 'image_id', 'laterality',
                    'view_position', 'label', 'img_path']].head(3).to_string())

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

# Pick one patient from VinDr to visualise all 4 views
sample_study = vindr_clean['study_id'].iloc[0]
sample       = vindr_clean[vindr_clean['study_id'] == sample_study]

# Get all 4 view paths
views = {
    'L-CC':  sample[(sample['laterality']=='LEFT')  & (sample['view_position']=='CC')]['img_path'].values[0],
    'L-MLO': sample[(sample['laterality']=='LEFT')  & (sample['view_position']=='MLO')]['img_path'].values[0],
    'R-CC':  sample[(sample['laterality']=='RIGHT') & (sample['view_position']=='CC')]['img_path'].values[0],
    'R-MLO': sample[(sample['laterality']=='RIGHT') & (sample['view_position']=='MLO')]['img_path'].values[0],
}

# Plot all 4 views
fig, axes = plt.subplots(1, 4, figsize=(16, 5))
fig.suptitle(f'Sample Patient — 4 Mammogram Views\n'
             f'Label: {"Malignant" if sample["label"].max()==1 else "Benign"}',
             fontsize=13, fontweight='bold')

for ax, (view_name, img_path) in zip(axes, views.items()):
    img = Image.open(img_path).convert('L')
    ax.imshow(img, cmap='gray')
    ax.set_title(view_name, fontsize=12, fontweight='bold')
    ax.axis('off')

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/cvaf_swinb/sample_4views_vindr.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved to Google Drive")

# Match CBIS images to their paths
The image file path column contains the SeriesUID

 Format: Mass-Training_P_00001_LEFT_CC/SeriesUID/000000.dcm

 We need the SeriesUID part to find the JPEG

 First load dicom_info.csv to get full mammogram series only


In [ ]:
dicom_info = pd.read_csv(f'{base}/cbis/csv/dicom_info.csv')

print("dicom_info shape:", dicom_info.shape)
print()
print("SeriesDescription values:")
print(dicom_info['SeriesDescription'].value_counts())

# Get full mammogram series only from dicom_info

In [ ]:
full_mammo = dicom_info[dicom_info['SeriesDescription'] == 'full mammogram images']
print(f"Full mammogram series: {len(full_mammo)}")


# Build index: SeriesUID → image path


In [ ]:
cbis_full_index = {}
for _, row in full_mammo.iterrows():
    img_path = str(row['image_path'])
    parts    = img_path.replace('\\', '/').strip().split('/')
    if len(parts) >= 3:
        series_uid = parts[-2]
        full_path  = f'{base}/cbis/jpeg/{series_uid}'
        if os.path.exists(full_path):
            files = [f for f in os.listdir(full_path) if f.endswith('.jpg')]
            if files:
                cbis_full_index[series_uid] = f'{full_path}/{files[0]}'

print(f"Full mammogram images indexed: {len(cbis_full_index)}")

# Match each CBIS row to its actual image file
 The image file path column has format:

 Mass-Training_P_00001_LEFT_CC/SeriesUID/000000.dcm

 We need the SeriesUID part



In [ ]:
def get_series_uid(file_path):
    parts = str(file_path).replace('\\', '/').strip().split('/')
    if len(parts) >= 2:
        return parts[-2]
    return None

cbis_clean['series_uid'] = cbis_clean['image file path'].apply(get_series_uid)
cbis_clean['img_path']   = cbis_clean['series_uid'].map(cbis_full_index)

# Check how many matched
matched   = cbis_clean['img_path'].notna().sum()
unmatched = cbis_clean['img_path'].isna().sum()

print(f"Matched images   : {matched}")
print(f"Unmatched images : {unmatched}")
print()
print("Sample rows:")
print(cbis_clean[['patient_id', 'left or right breast',
                   'image view', 'label', 'img_path']].head(3).to_string())

# Fix: make proper copy to avoid SettingWithCopyWarning
 Fix: remove 49 rows where image path could not be found


In [ ]:
cbis_clean = cbis_clean.copy()



# Add image paths


In [ ]:
cbis_clean['series_uid'] = cbis_clean['image file path'].apply(get_series_uid)
cbis_clean['img_path']   = cbis_clean['series_uid'].map(cbis_full_index)



# Remove rows with no image path


In [ ]:
cbis_clean = cbis_clean.dropna(subset=['img_path'])

print(f"Rows after removing unmatched : {len(cbis_clean)}")
print(f"Patients remaining            : {cbis_clean['patient_id'].nunique()}")
print(f"Benign (0)                    : {(cbis_clean['label']==0).sum()}")
print(f"Malignant (1)                 : {(cbis_clean['label']==1).sum()}")

# Check which patients still have all 4 views after removing unmatched rows


In [ ]:
required_views = {('LEFT','CC'), ('LEFT','MLO'), ('RIGHT','CC'), ('RIGHT','MLO')}

complete_ids = []
for patient_id, group in cbis_clean.groupby('patient_id'):
    views = set(zip(group['left or right breast'], group['image view']))
    if required_views.issubset(views):
        complete_ids.append(patient_id)



# Keep only complete patients


In [ ]:
cbis_clean = cbis_clean[cbis_clean['patient_id'].isin(complete_ids)].copy()

print(f"Patients with all 4 views : {cbis_clean['patient_id'].nunique()}")
print(f"Total rows                : {len(cbis_clean)}")
print(f"Benign (0)                : {(cbis_clean['label']==0).sum()}")
print(f"Malignant (1)             : {(cbis_clean['label']==1).sum()}")

# Outlier detection
We open each patient's L-CC image and check pixel statistics

 Too dark = mean pixel value too low

 Too bright = mean pixel value too high  

 No contrast = standard deviation too low

We are doing **outlier detection** — finding bad quality images.

---

### What we are doing step by step

**1. We open each patient's L-CC image**
The L-CC view is the left breast from above. We use this one image to check quality.

**2. We convert it to grayscale**
Grayscale means black and white — pixel values from 0 to 255.
- 0 = completely black
- 255 = completely white

**3. We calculate two numbers for each image:**

| Number | What it measures | Good range |
|---|---|---|
| **Mean** | Average brightness | Not too dark, not too bright |
| **Std** | Contrast — how much variation | Must be above a minimum |

**4. We use the 3-sigma rule to find outliers**

```
Normal range = average ± 3 × standard deviation

If mean < average - 3σ → image is TOO DARK  → outlier
If mean > average + 3σ → image is TOO BRIGHT → outlier
If std  < average - 3σ → image has NO CONTRAST → outlier
```

---

### Why do we remove outliers?

Bad images confuse the model. If you train on corrupted or too dark images the model learns the wrong patterns.

---

### Example of what a bad image looks like

```
Good image  : mean=120  std=45  (normal mammogram)
Too dark    : mean=10   std=5   (almost black — scanner error)
Too bright  : mean=250  std=3   (almost white — overexposed)
No contrast : mean=128  std=2   (completely grey — no detail)
```


#Scanning CBIS-DDSM images for outliers

In [ ]:
print("Scanning CBIS-DDSM images for outliers ...")
print("This may take a few minutes ...\n")

cbis_stats = []

for patient_id, group in cbis_clean.groupby('patient_id'):
    # Get L-CC image for this patient
    lcc = group[
        (group['left or right breast'] == 'LEFT') &
        (group['image view'] == 'CC')
    ]['img_path'].values

    if len(lcc) == 0:
        continue

    try:
        arr  = np.array(Image.open(lcc[0]).convert('L'))
        cbis_stats.append({
            'patient_id': patient_id,
            'label':      group['label'].max(),
            'source':     'cbis',
            'mean':       float(arr.mean()),
            'std':        float(arr.std()),
        })
    except Exception as e:
        print(f"  Error reading {patient_id}: {e}")

cbis_stats_df = pd.DataFrame(cbis_stats)
print(f"Scanned : {len(cbis_stats_df)} patients")
print()
print("Pixel statistics:")
print(f"  Mean : avg={cbis_stats_df['mean'].mean():.1f}  std={cbis_stats_df['mean'].std():.1f}")
print(f"  Std  : avg={cbis_stats_df['std'].mean():.1f}   std={cbis_stats_df['std'].std():.1f}")

# Find outliers using 3-sigma rule


In [ ]:
mean_mu  = cbis_stats_df['mean'].mean()
mean_sig = cbis_stats_df['mean'].std()
std_mu   = cbis_stats_df['std'].mean()
std_sig  = cbis_stats_df['std'].std()



# Calculate thresholds


In [ ]:
lower_mean = mean_mu - 3 * mean_sig
upper_mean = mean_mu + 3 * mean_sig
lower_std  = std_mu  - 3 * std_sig

print("Outlier thresholds:")
print(f"  Too dark    : mean < {lower_mean:.1f}")
print(f"  Too bright  : mean > {upper_mean:.1f}")
print(f"  No contrast : std  < {lower_std:.1f}")
print()



# Flag outliers


In [ ]:
cbis_stats_df['too_dark']    = cbis_stats_df['mean'] < lower_mean
cbis_stats_df['too_bright']  = cbis_stats_df['mean'] > upper_mean
cbis_stats_df['no_contrast'] = cbis_stats_df['std']  < lower_std
cbis_stats_df['is_outlier']  = (cbis_stats_df['too_dark'] |
                                cbis_stats_df['too_bright'] |
                                cbis_stats_df['no_contrast'])

print(f"Outliers found : {cbis_stats_df['is_outlier'].sum()}")
print(f"  Too dark     : {cbis_stats_df['too_dark'].sum()}")
print(f"  Too bright   : {cbis_stats_df['too_bright'].sum()}")
print(f"  No contrast  : {cbis_stats_df['no_contrast'].sum()}")

# Scan VinDr images for outliers


In [ ]:
print("Scanning VinDr images for outliers ...")
print("This may take a few minutes ...\n")

vindr_stats = []

for study_id, group in vindr_clean.groupby('study_id'):
    # Get L-CC image for this patient
    lcc = group[
        (group['laterality'] == 'LEFT') &
        (group['view_position'] == 'CC')
    ]['img_path'].values

    if len(lcc) == 0:
        continue

    try:
        arr = np.array(Image.open(lcc[0]).convert('L'))
        vindr_stats.append({
            'study_id': study_id,
            'label':    group['label'].max(),
            'source':   'vindr',
            'mean':     float(arr.mean()),
            'std':      float(arr.std()),
        })
    except Exception as e:
        print(f"  Error: {study_id}: {e}")

vindr_stats_df = pd.DataFrame(vindr_stats)
print(f"Scanned : {len(vindr_stats_df)} patients")
print()
print("Pixel statistics:")
print(f"  Mean : avg={vindr_stats_df['mean'].mean():.1f}  std={vindr_stats_df['mean'].std():.1f}")
print(f"  Std  : avg={vindr_stats_df['std'].mean():.1f}   std={vindr_stats_df['std'].std():.1f}")